# The Three-Qubit Bit-Flip Code

## The simplest quantum error-correcting code, capable of correcting a single $X$ (bit-flip) error. It is not a full quantum code — it doesn't catch phase errors — but it cleanly illustrates the **encode / detect / correct** pattern that all QEC codes use.

# 1. The code

## Encode one *logical* qubit into three physical qubits:

$$ \Large  |0\rangle_L = |000\rangle, \qquad |1\rangle_L = |111\rangle. $$

## A general logical state $\alpha|0\rangle_L + \beta|1\rangle_L = \alpha|000\rangle + \beta|111\rangle$ is *not* a clone of $|0\rangle$ — that would violate no-cloning. It is an entangled state of three qubits with the *information* redundantly encoded in their correlations.

# 2. Encoding circuit

## From $(\alpha|0\rangle + \beta|1\rangle)|0\rangle|0\rangle$, apply CNOT(0→1) then CNOT(0→2):

$$ \Large  |\psi\rangle |00\rangle \;\to\; \alpha|000\rangle + \beta|111\rangle. $$

## This is the unique way to encode without violating no-cloning.

# 3. Syndrome measurement

## Suppose a single $X$ error has occurred on qubit $i$. The encoded state becomes one of:

| Error  | State                                  |
|--------|----------------------------------------|
| $X_0$  | $\alpha\lvert100\rangle + \beta\lvert011\rangle$ |
| $X_1$  | $\alpha\lvert010\rangle + \beta\lvert101\rangle$ |
| $X_2$  | $\alpha\lvert001\rangle + \beta\lvert110\rangle$ |
| none   | $\alpha\lvert000\rangle + \beta\lvert111\rangle$ |

## We can distinguish these by measuring the **parity** of pairs of qubits, *without* learning $\alpha$ or $\beta$:

## - Measure $Z_0 Z_1$ — outcome $+1$ if qubits 0 and 1 agree, $-1$ if they disagree.
## - Measure $Z_1 Z_2$ — similarly for qubits 1 and 2.

## The pair of outcomes is called the **syndrome**: $(00)$ means no error; $(10)$ means $X_0$; $(11)$ means $X_1$; $(01)$ means $X_2$.

In [1]:
# Simulate the 3-qubit code, injecting a single X error and recovering.
import numpy as np

ket0, ket1 = np.array([1, 0]), np.array([0, 1])
X = np.array([[0,1],[1,0]], dtype=complex)
I = np.eye(2, dtype=complex)

def encode(alpha, beta):
    psi = alpha * np.kron(np.kron(ket0, ket0), ket0) + beta * np.kron(np.kron(ket1, ket1), ket1)
    return psi

def apply(op, i, n=3):
    ops = [I]*n; ops[i] = op
    U = ops[0]
    for o in ops[1:]:
        U = np.kron(U, o)
    return U

def syndrome(rho):
    """Return (s1, s2) with s1 = parity(q0,q1), s2 = parity(q1,q2)."""
    # Project onto subspaces of fixed parity
    sx = 0
    for basis in range(8):
        bits = [(basis>>k)&1 for k in (2,1,0)]
        if abs(rho[basis])**2 > 1e-9:
            s1 = bits[0]^bits[1]
            s2 = bits[1]^bits[2]
            return (s1, s2)

alpha, beta = np.sqrt(0.3), np.sqrt(0.7)
for i in (0, 1, 2):
    encoded = encode(alpha, beta)
    corrupted = apply(X, i) @ encoded
    syn = syndrome(corrupted)
    correction_map = {(0,0): None, (1,0): 0, (1,1): 1, (0,1): 2}
    j = correction_map[syn]
    recovered = apply(X, j) @ corrupted if j is not None else corrupted
    print(f'X on q{i}: syndrome = {syn}, correct q{j}, recovered matches? {np.allclose(recovered, encoded)}')

X on q0: syndrome = (1, 0), correct q0, recovered matches? True
X on q1: syndrome = (1, 1), correct q1, recovered matches? True
X on q2: syndrome = (0, 1), correct q2, recovered matches? True


# 4. What the code can't do

## - **Phase flips**: $Z$ on any single qubit takes $\alpha|000\rangle + \beta|111\rangle \to \alpha|000\rangle - \beta|111\rangle$,   which has trivial syndrome but is a different logical state. The bit-flip code is invisible to phase errors.
## - **Two errors**: if errors happen on two qubits simultaneously, the syndrome is misinterpreted and the recovery makes things worse.

## These motivate richer codes — first the Shor 9-qubit code that catches both flips and phases.

# Recap

## - Encode $|0\rangle_L = |000\rangle$, $|1\rangle_L = |111\rangle$.
## - Syndrome from parity measurements $Z_0 Z_1$ and $Z_1 Z_2$ locates the bit flip.
## - Corrects 1 $X$ error; misses $Z$ errors.
## - Pattern (encode / measure syndrome / apply correction) reappears in every QEC code.